In [1]:
import pandas as pd
import PyPDF2
import os
import seaborn as sns
import openai
import matplotlib.pyplot as plt
import numpy as np
import re
from textstat import flesch_reading_ease
import requests
import json
import pandas as pd
import prettytable
import zipfile
import difflib
from difflib import get_close_matches
from sklearn.preprocessing import OneHotEncoder
from scipy.stats import entropy
import pyarrow.parquet as pq


In [2]:
### DataFrame and Graphs Settings
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colum', None)
pd.set_option("display.max_colwidth", None)

sns.set(font_scale=1.5)
sns.set(style='white', font_scale=1.5)

## Main

In [3]:
df= pd.read_parquet('/Users/mgor/Dropbox/Second YPP/Data/Data Aux/compustat_database.parquet')


In [4]:
df.head()

,gvkey,tic,cusip,comCounty,comDom,comCountry,comNAICS,comHeadquarters,OccFam,OccFamName,SOC,SOCName,CanonEmployer,Sector,SectorName,NAICS3,NAICS4,NAICS5,NAICS6,State,County,FIPSState,FIPSCounty,FIPS,BestFitMSA,BestFitMSAName,Year,qtr,Degree,Exp,boncom,parttime,NoWage,MeanWage,VarRange,hires_nt,jobOpenings_nt,layoffs_nt,otherSeparations_nt,quits_nt,totalSeparations_nt,unemployedPerOpening_nt,hires_st,jobOpenings_st,layoffs_st,quits_st,totalSeparations_st,unemployedPerOpening_st,tot_emp,emp_prse,jobs_1000,natEmp,a_mean_nt,SOCShare,0507SOCShare,shiftSOC,wkWageNAICS,shiftNAICS,07LNAICSShare,BShareimq,BShareHK,DivIntRent,UIC,earnWork,emplGovContr,emplPensionContr,farmPropInc,incManBen,netEarnbyRes,nonfarmPropInc,pInc,pTransReceipt,percapDivRent,percapEarn,percapInc,percapIncMan,percapRet,percapTrans,percapUIC,pop,propInc,retirement,supplWages,wages,lgdp_change,igdpq_ch,crisis_ch,lpctunion,pctunion_ind,pctunion_occ,employment_st,laborForce_st,unemployment_st,unemploymentRate_st,laborForce_msa,employment_msa,unemployment_msa,unemploymentRate_msa,employment_ct,laborForce_ct,unemployment_ct,unemploymentRate_ct,MSAopenings_employer,FIPSopenings_employer,MSAOccopenings_employer,FIPSOccopenings_employer,MSASectoropenings_employer,FIPSSectoropenings_employer,TopQuartileMSAopenings,TopQuartileFIPSopenings,TopQuartileMSAOccopenings,TopQuartileFIPSOccopenings,TopQuartileMSASectoropenings,TopQuartileFIPSSectoropenings,tight,vacan,tightocc,vacanocc,tightsector,vacansector,Male,Female,white,black,americanIndian,alaskaNative,asian,hawaiianPacificIslander,someOtherRace,ageMean,pop_msa,white_msa,black_msa,americanIndianAlaskaNative_msa,asian_msa,nativeHawaiianPacificIslander_msa,someOtherRace_msa,kcosts,kprod,kshare,ktolb,hworked,lcomp,lprod,lshare,opw,routput,tfp,ulc,intmig,dommig,college_st,nonprofit,quarter,roa,roe,logSize,gProfit,tobinq,lev,comAge,IsSpecialized,IsBaseline,IsSoftware,Administration,agricultureHorticulture,Analysis,architectureConstruction,Business,customerClient,Design,economicsPolicy,educationTraining,energyUtilities,Engineering,Environment,Finance,healthCare,humanResources,industryKnowledge,informationTechnology,Legal,maintenanceRepair,manufacturingProduction,marketingPublic,mediaWriting,personalCare,publicSafety,Religion,Sales,scienceResearch,supplyChain,notMapped,SkillsCount,SkillClusterCount,SkillClusterbyFam,SkillFamilyCount,ClusterToFamilyRatio,SkillFamilyCoverage,SkillFamilyEntropy,pecuniary_benefits_count,job_design_characteristics_count,career_development_opportunities_count,organizational_culture_count,meaningful_work_count,environmental_initiatives_count,social_initiatives_count,job_tasks_requirements_count,main_count,length,sr_count,pecuniary_benefits_prop,job_design_characteristics_prop,career_development_opportunities_prop,organizational_culture_prop,meaningful_work_prop,environmental_initiatives_prop,social_initiatives_prop,job_tasks_requirements_prop,main_prop,sr_prop,opEmptot_count,bartik,sr_jobs,main_jobs,sr_ind,main_ind
0,10005.0,SRCTQ,853887206,None,D,USA,323111,OH,11,Management Occupations,11-2022,Sales Managers,Standard Register,54,"Professional, Scientific, and Technical Services",-999,-999,-999,-999,Ohio,Montgomery,39,113,39113,19380,"Dayton, OH",2010,1,1.0,8.0,0.0,0.0,1.0,NaN,NaN,3.3,2.0,1.5,0.2,1.4,3.1,5.7,2.8,1.9,1.5,1.3,3.0,6.5,790.0,11.1,2.191,319300.0,114110.0,0.002231,0.001551,-0.020082,1176.0,-0.037028,0.000000,-0.000103,-0.004842,-1.493404,-4.496601,1.009402,-0.970814,3.426168,5.405997,8.991433,1.031472,8.879074,1.857545,6.505426,-1.577592,0.949260,1.770415,8.955224,6.825490,6.416065,-4.625551,0.084440,8.859660,6.918872,2.009323,0.174402,-0.002065,NaN,NaN,0.072964,0.017361,0.009060,5259913.0,5904597.0,644684.0,10.9,12.912933,12.788313,10.768780,11.7,12.363196,12.495378,10.406442,12.4,1.386294,1.386294,0.693147,0.693147,1.098612,1.098612,0.0,0.0,0.0,0.0,0.0,0.0,-5.309194,-7.453347,-6.625645,-8.769798,-6.680006,-8.824159,55.582865,44.417135,94.279611,2.999181,0.000000,0.0,1.629799,0.000000,0.26626

In [20]:
count = df[(df['gvkey'] == '-999') & (df['cusip']!= '-999')].shape[0]
print(f"Number of rows where 'col1' is -999 and 'col2' is NaN: {count}")


Number of rows where 'col1' is -999 and 'col2' is NaN: 0


In [29]:
groupby_cols = ['gvkey', 'tic', 'cusip', 'comCounty', 'comDom', 'comCountry', 'comNAICS', 'comHeadquarters', 
        'OccFam', 'OccFamName', 'SOC', 'SOCName', 'CanonEmployer', 'Sector', 'SectorName', 
        'NAICS3', 'NAICS4', 'NAICS5', 'NAICS6', 'State', 'County', 
        'FIPSState', 'FIPSCounty', 'FIPS', 'BestFitMSA', 'BestFitMSAName', 'Year', 'qtr']

In [30]:
numeric_cols = [col for col in df.columns if col not in groupby_cols]
df[numeric_cols] = df[numeric_cols].replace(-999, np.nan)

In [31]:
df['bartik'] = df['BShareimq']*df['shiftNAICS']
df['MeanWage'] = np.log(df['MeanWage'])
df = df.dropna(subset=['tight', 'bartik'])


In [7]:
df.dtypes

gvkey                                      object
tic                                        object
cusip                                      object
comCounty                                  object
comDom                                     object
comCountry                                 object
comNAICS                                    int64
comHeadquarters                            object
OccFam                                      int64
OccFamName                                 object
SOC                                        object
SOCName                                    object
CanonEmployer                              object
Sector                                      int64
SectorName                                 object
NAICS3                                      int64
NAICS4                                      int64
NAICS5                                      int64
NAICS6                                      int64
State                                      object


In [17]:
df.tight.describe()

count    563357.000000
mean        -19.938021
std         124.154600
min        -999.000000
25%          -5.516107
50%          -4.062753
75%          -2.805379
max           0.977351
Name: tight, dtype: float64

## Merge Text Measures - Main

In [126]:
df_txt= pd.read_parquet('/Users/mgor/Dropbox/Second YPP/Data/Data Aux/US_XML_AddFeed_20210701_20210707.parquet')

In [127]:
df_txt.head()

JobID                 CanonEmployer     JobDate  \
0  39032229262  Lindley Park Filling Station  2021-06-30   
1  39032229346              Northrop Grumman  2021-06-30   
2  39032229414                Uniform Outlet  2021-06-30   
3  39032229428                Hallmark Cards  2021-06-30   
4  39032229249             Prime Hospitality  2021-06-30   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                

In [64]:
df = df.dropna(subset = ['JobText'])
df= df[(df['CanonEmployer'].notna()) & (df['CanonEmployer'] != None) & (df['CanonEmployer'] != "None")]
df = df.drop(columns=['IsDuplicate','IsDuplicateOf', 'JobReferenceID'])

# Date
df['year'] = df.JobDate.apply(lambda x: x[0:4])
df['month'] = df.JobDate.apply(lambda x: x[5:7])

In [91]:

df_txt=df_txt.rename(columns={'JobID':'BGTJobId'}, errors='ignore')
df_txt.drop(columns=['CanonEmployer', 'JobText', 'CleanText','year'], inplace=True, errors='ignore')

In [55]:
df_txt['month']=df_txt['JobDate'].str[5:7].astype(int)  


In [123]:
df_txt=pd.read_parquet('/Users/mgor/Dropbox/Second YPP/Data/Data Aux/Text_2010-12.parquet')

In [56]:
df_merge=pd.merge(df, df_txt, on=['BGTJobId'], how='left',indicator=True)

In [67]:
df.month.value_counts()

11    33242
12    27376
Name: month, dtype: int64

## Merge Skills-Main

In [82]:
df= pd.read_parquet('/Users/mgor/Dropbox/Second YPP/Data/Data Aux/Main_2014-12.parquet')

In [97]:
df_skill= pd.read_parquet('/Users/mgor/Dropbox/Second YPP/Data/Data Aux/Skills_2018-01.parquet')

In [93]:
df_txt['BGTJobId']=pd.to_numeric(df_txt['BGTJobId'], errors='coerce').astype(int)


In [96]:
df_skill.dtypes

BGTJobId                      int64
IsSpecialized               float64
IsBaseline                  float64
IsSoftware                  float64
Administration              float64
agricultureHorticulture     float64
Analysis                    float64
architectureConstruction    float64
Business                    float64
customerClient              float64
Design                      float64
economicsPolicy             float64
educationTraining           float64
energyUtilities             float64
Engineering                 float64
Environment                 float64
Finance                     float64
healthCare                  float64
humanResources              float64
industryKnowledge           float64
informationTechnology       float64
Legal                       float64
maintenanceRepair           float64
manufacturingProduction     float64
marketingPublic             float64
mediaWriting                float64
personalCare                float64
publicSafety                

In [63]:
df=df.drop(columns=['CanonTitle','BGTOccGroupName2','BGTCareerAreaName2','MSAName'], errors='ignore')

In [64]:
df = df.drop(columns=[col for col in df.columns if col.endswith('_y')], errors='ignore')
df = df.rename(columns={col: col[:-2] for col in df.columns if col.endswith('_x')})

In [89]:
df=df.drop_duplicates(subset='BGTJobId')

In [90]:
df_aux=pd.merge(df, df_skill, on=['BGTJobId'], how='inner', indicator=True)

## Skills

In [76]:
df= pd.read_table('/Users/mgor/Dropbox/Second YPP/Data/Data Aux/Skills_2010-01.zip', encoding='latin')

In [77]:
# Initialize the encoder
encoder = OneHotEncoder(sparse=True, dtype=int)
# Fit and transform the categorical column
encoded = encoder.fit_transform(df[['SkillClusterFamily']])

# Create a DataFrame from the encoded sparse matrix
encoded_df = pd.DataFrame.sparse.from_spmatrix(
    encoded, columns=encoder.get_feature_names_out(['SkillClusterFamily'])
)

# Rename the columns to remove the prefix
encoded_df.columns = [col.replace('SkillClusterFamily_', '') for col in encoded_df.columns]

# Concatenate the encoded DataFrame with the original DataFrame
df = pd.concat([df.reset_index(drop=True), encoded_df.reset_index(drop=True)], axis=1)

In [78]:
def rename_column(col_name):
        # Special case for 'na'
        if col_name.lower() == 'na':
            return 'notMapped'
        
        # Process columns with spaces only
        if ' ' in col_name:
            # Remove commas and "and"
            cleaned_name = re.sub(r',', '', col_name)
            cleaned_name = re.sub(r'\band\b', '', cleaned_name, flags=re.IGNORECASE)
            
            # Split into words
            words = cleaned_name.split()
            
            # Take the first two words only and apply camelCase
            if len(words) > 1:
                return words[0].lower() + words[1].capitalize()
            else:
                return words[0].lower()
        
        # Return original name for other cases
        return col_name

# Apply renaming logic
df = df.rename(columns={col: rename_column(col) for col in df.columns})
    

In [79]:
df.head()

,BGTJobId,JobDate,Skill,SkillCluster,SkillClusterFamily,IsSpecialized,IsBaseline,IsSoftware,Salary,Administration,agricultureHorticulture,Analysis,architectureConstruction,Business,customerClient,Design,economicsPolicy,educationTraining,energyUtilities,Engineering,Environment,Finance,healthCare,humanResources,industryKnowledge,informationTechnology,Legal,maintenanceRepair,manufacturingProduction,marketingPublic,mediaWriting,personalCare,publicSafety,Religion,Sales,scienceResearch,supplyChain,notMapped
0,289187887,2010-01-11,Customer Service,Basic Customer Service,Customer and Client Support,1,0,0,-999,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,289187887,2010-01-11,Sales Management,Sales Management,Sales,1,0,0,-999,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0
2,289187887,2010-01-11,Online Sales,Online Sales,Sales,1,0,0,-999,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0
3,289187887,2010-01-11,Cash Handling,Cash Register Operation,Customer and Client Support,1,0,0,-999,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,289187887,2010-01-11,Sales,General Sales,Sales,1,0,0,-999,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0


In [80]:
df['SkillsCount'] = df.groupby('BGTJobId')['Skill'].transform('nunique')
df['SkillClusterCount'] = df.groupby('BGTJobId')['SkillCluster'].transform('nunique')
df['SkillClusterbyFam'] = df.groupby(['BGTJobId', 'SkillClusterFamily'])['SkillCluster'].transform('nunique')
df['SkillFamilyCount'] = df.groupby('BGTJobId')['SkillClusterFamily'].transform('nunique')
df['ClusterToFamilyRatio'] = df['SkillClusterbyFam'] / df['SkillFamilyCount']

total_families = df['SkillClusterFamily'].nunique()
df['SkillFamilyCoverage'] = (
    df['SkillFamilyCount'] / total_families
)

In [8]:
def calculate_entropy(series):
    counts = series.value_counts(normalize=True)
    return entropy(counts)

df['SkillFamilyEntropy'] = df.groupby('BGTJobId')['SkillClusterFamily'].transform(calculate_entropy)

In [81]:
df.drop(columns=['JobDate', 'Skill', 'SkillCluster', 'SkillClusterFamily', 'Salary'], inplace=True)

In [82]:
# Convert only sparse columns to dense
sparse_cols = [col for col in df.columns if pd.api.types.is_sparse(df[col])]
df[sparse_cols] = df[sparse_cols].sparse.to_dense()



In [83]:
df.head()

,BGTJobId,IsSpecialized,IsBaseline,IsSoftware,Administration,agricultureHorticulture,Analysis,architectureConstruction,Business,customerClient,Design,economicsPolicy,educationTraining,energyUtilities,Engineering,Environment,Finance,healthCare,humanResources,industryKnowledge,informationTechnology,Legal,maintenanceRepair,manufacturingProduction,marketingPublic,mediaWriting,personalCare,publicSafety,Religion,Sales,scienceResearch,supplyChain,notMapped,SkillsCount,SkillClusterCount,SkillClusterbyFam,SkillFamilyCount,ClusterToFamilyRatio,SkillFamilyCoverage
0,289187887,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,8,7,2,3,0.666667,0.103448
1,289187887,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,8,7,4,3,1.333333,0.103448
2,289187887,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,8,7,4,3,1.333333,0.103448
3,289187887,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,8,7,2,3,0.666667,0.103448
4,289187887,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,8,7,4,3,1.333333,0.103448


In [84]:
df = df.groupby('BGTJobId').mean().reset_index()

In [13]:
df.to_parquet('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/test.parquet', compression='snappy', index=False)

In [85]:
len(df)

260088